In [1]:
# install required libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np

In [2]:
# checking if the GPU is enabled
print(torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

True


In [3]:
# using Google Drive to save checkpoints
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Training Pipeline
- Extracted an Embedding described in F.1
- Trained a single linear layer of size d x C
- d is feature dimensions, C is the number of classes

(single layer) increase the initial learning rate by a factor of 100 to 2.5

multiply the number of epochs by 2 (to 1000)

In [4]:
# transformations
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomResizedCrop(size=32),
    transforms.RandomApply([transforms.ColorJitter(brightness=0.8, contrast=0.8, saturation=0.8, hue=0.2)], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

embed_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

In [5]:
# generates two augmented views of the same image, for SimCLR contrastive learning
class SimCLRDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, transform):
        self.dataset = dataset
        self.transform = transform

    def __getitem__(self, idx):
        x, _ = self.dataset[idx]
        x1 = self.transform(x)
        x2 = self.transform(x)
        return x1, x2

    def __len__(self):
        return len(self.dataset)

In [6]:
# loading CIFAR-10 dataset, and wrapping it
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
dataset = SimCLRDataset(trainset, transform)
loader = torch.utils.data.DataLoader(dataset, batch_size=512, shuffle=True, num_workers=4, drop_last=True) # every batch is of 512 size

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


# SimCLR

In [7]:
# pretrained on ImageNet
# reducing kernel, stride and maxpool to match up with CIFAR-10s image size
resnet18 = torchvision.models.resnet18(weights=None, progress=True)
resnet18.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
resnet18.maxpool = nn.Identity()
resnet18.fc = nn.Identity()

In [8]:
# only used during training
class ProjectionHead(nn.Module):
  def __init__(self):
    super().__init__()
    self.fc1 = nn.Linear(512, 512)
    self.bn = nn.BatchNorm1d(512)
    self.relu = nn.ReLU(inplace=True)
    self.fc2 = nn.Linear(512, 128)

  def forward(self, x):
    x = self.fc1(x)
    x = self.bn(x)
    x = self.relu(x)
    x = self.fc2(x)
    return F.normalize(x, dim=1)

In [9]:
# SimCLR - wrapping ResNet18 and the projection head
class SimCLR(nn.Module):
  def __init__(self):
    super().__init__()
    self.encoder = resnet18
    self.projection_head = ProjectionHead()

  def forward(self, x1, x2): #two images
      z1 = self.encoder(x1)
      z2 = self.encoder(x2)

      p1 = self.projection_head(z1)
      p2 = self.projection_head(z2)

      return p1, p2

  def embedding(self, x):
    return self.encoder(x)

Loss

In [10]:
# contrastive loss
def nt_xent_loss(z1, z2, temperature=0.5):
    N = z1.size(0)

    z = torch.cat([z1, z2], dim=0)  # (2N, 128)

    similarity = F.cosine_similarity(z.unsqueeze(1), z.unsqueeze(0), dim=2)
    similarity /= temperature

    # extract positives from the FULL matrix before masking
    positives = torch.cat([
        torch.diag(similarity, N),
        torch.diag(similarity, -N)
    ], dim=0)  # (2N,)

    # mask self-similarity
    mask = torch.eye(2*N, dtype=torch.bool).to(z.device)
    similarity = similarity.masked_fill(mask, float('-inf'))

    logits = similarity  # (2N, 2N)
    labels = torch.cat([
        torch.arange(N, 2*N),  # first N samples match with last N
        torch.arange(0, N)     # last N samples match with first N
    ]).to(z.device)

    loss = F.cross_entropy(logits, labels)
    return loss

In [11]:
model = SimCLR().to(device)

In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.4, momentum=0.9, weight_decay=1e-4)
#T_max should be num_epochs
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=500, eta_min=0)

In [13]:
dummy = torch.randn(2, 3, 32, 32).to(device)
h = model.encoder(dummy)
print(h.shape)  # should be torch.Size([2, 512])

torch.Size([2, 512])


Training model

In [15]:
# for loading a checkpoint only..!
"""
checkpoint_path = '/content/drive/MyDrive/5CCSAMLF/FS_Embed/models/simclr_latest.pth'
checkpoint = torch.load(checkpoint_path)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
start_epoch = checkpoint['epoch'] + 1

print(f"Loaded checkpoint from epoch {start_epoch}, loss: {checkpoint['loss']:.4f}")
"""

'\ncheckpoint_path = \'/content/drive/MyDrive/5CCSAMLF/FS_Embed/models/simclr_latest.pth\'\ncheckpoint = torch.load(checkpoint_path)\nmodel.load_state_dict(checkpoint[\'model_state_dict\'])\noptimizer.load_state_dict(checkpoint[\'optimizer_state_dict\'])\nscheduler.load_state_dict(checkpoint[\'scheduler_state_dict\'])\nstart_epoch = checkpoint[\'epoch\'] + 1\n\nprint(f"Loaded checkpoint from epoch {start_epoch}, loss: {checkpoint[\'loss\']:.4f}")\n'

In [ ]:
checkpoint_path = '/content/drive/MyDrive/5CCSAMLF/FS_Embed/models/simclr_latest.pth'
num_epochs = 500
best_loss = 1000

for epoch in range(num_epochs): # or (start_epochs, num_epochs)
    model.train()
    running_loss = 0
    for (x1, x2) in loader:
        x1 = x1.to(device)
        x2 = x2.to(device)

        # Forward pass
        z1, z2 = model(x1, x2)

        loss = nt_xent_loss(z1, z2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()

    avg_loss = running_loss / len(loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

    # save the best model
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save({'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'loss': avg_loss},
                   checkpoint_path)
        print(f"Saved best model - epoch {epoch+1} with loss: {avg_loss:.4f}")

Epoch 1/500, Loss: 6.5511
Saved best model - epoch 1 with loss: 6.5511
Epoch 2/500, Loss: 6.3457
Saved best model - epoch 2 with loss: 6.3457
Epoch 3/500, Loss: 6.2070
Saved best model - epoch 3 with loss: 6.2070
Epoch 4/500, Loss: 6.0865
Saved best model - epoch 4 with loss: 6.0865
Epoch 5/500, Loss: 6.0238
Saved best model - epoch 5 with loss: 6.0238
Epoch 6/500, Loss: 5.9703
Saved best model - epoch 6 with loss: 5.9703
Epoch 7/500, Loss: 5.9076
Saved best model - epoch 7 with loss: 5.9076
Epoch 8/500, Loss: 5.8589
Saved best model - epoch 8 with loss: 5.8589
Epoch 9/500, Loss: 5.8233
Saved best model - epoch 9 with loss: 5.8233
Epoch 10/500, Loss: 5.7984
Saved best model - epoch 10 with loss: 5.7984
Epoch 11/500, Loss: 5.7743
Saved best model - epoch 11 with loss: 5.7743
Epoch 12/500, Loss: 5.7525
Saved best model - epoch 12 with loss: 5.7525
Epoch 13/500, Loss: 5.7393
Saved best model - epoch 13 with loss: 5.7393
Epoch 14/500, Loss: 5.7219
Saved best model - epoch 14 with loss: 5.7

extracting embeddings

In [ ]:
# loading trained model once training has stopped
checkpoint_path = '/content/drive/MyDrive/5CCSAMLF/FS_Embed/models/simclr_latest.pth'
checkpoint = torch.load(checkpoint_path)

model = SimCLR().to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

In [ ]:
# 50k imgs
cifar_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=embed_transform)
cifar_trainloader = torch.utils.data.DataLoader(cifar_train, batch_size=512, shuffle=False)
# 10k imgs
cifar_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=embed_transform)
cifar_testloader = torch.utils.data.DataLoader(cifar_test, batch_size=512, shuffle=False)

In [ ]:
# extracting train embeddings
train_embeddings = []
train_labels = []

with torch.no_grad():
    for imgs, labels in cifar_trainloader:
        imgs = imgs.to(device)
        emb = model.embedding(imgs)
        emb = F.normalize(emb, dim=1)
        train_embeddings.append(emb.cpu())
        train_labels.append(labels)
        print(f"Processed {len(train_embeddings) * 512} images")

train_embeddings_tensor = torch.cat(train_embeddings, dim=0)
train_labels_tensor = torch.cat(train_labels, dim=0)     # ground truth

In [ ]:
# save train embeddings
torch.save({
    'embeddings': train_embeddings_tensor,  # (50000, 512) tensor
    'labels': train_labels_tensor           # (50000,) tensor
}, '/content/drive/MyDrive/5CCSAMLF/FS_Embed/models/train_embeddings.pth')

test..

In [ ]:
# extracting test embeddings
test_embeddings = []
test_labels = []

with torch.no_grad():
    for imgs, labels in cifar_testloader:
        imgs = imgs.to(device)
        emb = model.embedding(imgs)
        emb = F.normalize(emb, dim=1)
        train_embeddings.append(emb.cpu())
        train_labels.append(labels)
        print(f"Processed {len(test_embeddings) * 512} images")

test_embeddings_tensor = torch.cat(test_embeddings, dim=0)
test_labels_tensor = torch.cat(test_labels, dim=0)     # ground truth

In [ ]:
# save test embeddings
torch.save({
    'embeddings': test_embeddings_tensor,  # (50000, 512) tensor
    'labels': test_labels_tensor           # (50000,) tensor
}, '/content/drive/MyDrive/5CCSAMLF/FS_Embed/models/train_embeddings.pth')